# 02 — Feature Engineering
## Credit Risk: Loss Given Default (LGD) Prediction

**Objectives:**
- Construct LTV at default from origination LTV + HPI adjustment
- Engineer macroeconomic features (HPI change, unemployment)
- Encode categorical variables (property type, occupancy, channel, state → region)
- Validate feature distributions and directional relationships with LGD
- Save feature matrix for model training

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config import load_config
from src.data.features import (
    compute_ltv_at_default,
    add_hpi_change,
    add_unemployment,
    add_region,
    add_modification_flag_numeric,
    encode_categoricals,
    build_feature_matrix,
    STATE_TO_REGION,
)

config = load_config()
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
print('Setup complete')

## 1. Load Cleaned Data

In [ ]:
cleaned_path = Path(config.data.processed_dir) / 'cleaned.parquet'
if cleaned_path.exists():
    df = pd.read_parquet(cleaned_path)
    print(f'Loaded {len(df):,} rows')
    print(df.columns.tolist())
else:
    print('Run preprocessing pipeline first: python src/data/preprocess.py')

## 2. LTV at Default

LTV at default is the most predictive single feature. It reflects actual collateral coverage at the time of default, accounting for principal paydown and property price changes since origination.

```
LTV_at_default = current_UPB / estimated_property_value_at_default × 100

estimated_value = orig_UPB / (orig_LTV/100) × (1 + HPI_change)
```

In [ ]:
# TODO: Compute LTV at default and examine distribution
df = add_hpi_change(df)  # Must come before compute_ltv_at_default
df = compute_ltv_at_default(df)

print('LTV at default statistics:')
print(df['ltv_at_default'].describe())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['ltv_at_default'].clip(0, 200), bins=50, color='#2563EB', alpha=0.8, edgecolor='white')
ax.axvline(100, color='red', linestyle='--', label='LTV = 100% (underwater)')
ax.set_xlabel('LTV at Default (%)')
ax.set_ylabel('Count')
ax.set_title('Distribution of LTV at Default')
ax.legend()
plt.tight_layout()
plt.show()

pct_underwater = (df['ltv_at_default'] > 100).mean() * 100
print(f'Underwater at default (LTV > 100%): {pct_underwater:.1f}%')

## 3. LTV vs LGD Relationship

Higher LTV at default should correlate with higher LGD. This is the most important directional sanity check.

In [ ]:
# TODO: Binned LTV vs mean LGD
df['ltv_bin'] = pd.cut(df['ltv_at_default'].clip(0, 200), bins=10)
ltv_lgd = df.groupby('ltv_bin', observed=True)['loss_given_default'].agg(['mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(
    range(len(ltv_lgd)), 
    ltv_lgd['mean'],
    color='#2563EB', alpha=0.8,
    width=0.8
)
ax.set_xticks(range(len(ltv_lgd)))
ax.set_xticklabels([str(b) for b in ltv_lgd['ltv_bin']], rotation=45, ha='right')
ax.set_ylabel('Mean LGD')
ax.set_xlabel('LTV at Default Bin')
ax.set_title('Mean LGD by LTV at Default Decile — Expected: Monotonically Increasing')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. HPI Change Feature

In [ ]:
# TODO: Examine HPI change distribution and its effect on LGD
print('HPI change statistics (0.0 = placeholder, no external HPI joined yet):')
print(df['hpi_change'].describe())

# When HPI data is joined, plot: binned HPI change vs mean LGD
# Expected: negative HPI change → higher LGD

## 5. Regional Encoding

In [ ]:
# TODO: State → region mapping and mean LGD by region
df = add_region(df)

if 'region' in df.columns:
    region_stats = df.groupby('region')['loss_given_default'].agg(['mean', 'count', 'std']).reset_index()
    region_stats = region_stats.sort_values('mean', ascending=True)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(region_stats['region'], region_stats['mean'], color='#2563EB', alpha=0.8)
    ax.set_xlabel('Mean LGD')
    ax.set_title('Mean LGD by Census Region')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(region_stats.to_string(index=False))

## 6. Categorical Encoding

In [ ]:
# TODO: Label encode categoricals and examine encoded values
df_encoded, encoders = encode_categoricals(df, config.features.categorical)

print('Categorical encoding complete. Unique values per column:')
for col in config.features.categorical:
    if col in df_encoded.columns:
        n_unique = df_encoded[col].nunique()
        print(f'  {col}: {n_unique} unique encoded values')
        if col in encoders:
            print(f'    Classes: {list(encoders[col].classes_)}')

## 7. Feature Matrix

In [ ]:
# TODO: Build and inspect final feature matrix
X, y = build_feature_matrix(df_encoded, config)

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print('\nFeature summary:')
print(X.describe())

## 8. Feature Correlation Heatmap

In [ ]:
# TODO: Correlation heatmap of feature matrix
full_df = pd.concat([X, y], axis=1)

fig, ax = plt.subplots(figsize=(14, 12))
corr_matrix = full_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Key Findings

*To be completed after analysis:*

1. **LTV at default distribution:** [e.g., X% of defaulted loans were underwater]
2. **LTV → LGD relationship:** [Confirm monotonic relationship]
3. **Regional variation:** [Which regions have highest/lowest LGD]
4. **Feature correlations:** [Any multicollinearity concerns]
5. **Encoding decisions:** [Categorical encoding approach and rationale]